# Submit a Command Job

Generate a taxi-data summarization script, run it on the configured Azure ML compute, and inspect its terminal status and output URI.

**Sources:** Adapted from this repository's command-job patterns and the [Azure ML single-step examples](https://github.com/Azure/azureml-examples/tree/main/sdk/python/jobs/single-step), MIT License.

In [3]:
from pathlib import Path
import ast
import os
import textwrap

from azure.ai.ml import Input, MLClient, Output, command
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import ManagedIdentityConfiguration
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
COMPUTE_IDENTITY_CLIENT_ID = os.environ["AZUREML_COMPUTE_IDENTITY_CLIENT_ID"].strip()
if not COMPUTE_IDENTITY_CLIENT_ID:
    raise ValueError("AZUREML_COMPUTE_IDENTITY_CLIENT_ID must identify the compute cluster UMI")
EXPERIMENT_NAME = os.environ["WORKSHOP_EXPERIMENT_NAME"]
SUBMIT = os.getenv("SUBMIT_FOUNDATION_JOB", "false").lower() in {"1", "true", "yes"}

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [4]:
generated_code_dir = WORKSHOP_ROOT / "outputs/generated/foundations/command_job/code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
script_path = generated_code_dir / "summarize.py"
script_source = r'''
import argparse
import json
from pathlib import Path

import pandas as pd


def resolve_csv(path):
    path = Path(path)
    if path.is_file():
        return path
    matches = sorted(path.rglob("*.csv"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one CSV beneath {path}, found {len(matches)}")
    return matches[0]


parser = argparse.ArgumentParser(description="Summarize a taxi CSV")
parser.add_argument("--input-data", required=True)
parser.add_argument("--output-data", required=True)
args = parser.parse_args()
source_path = resolve_csv(args.input_data)
frame = pd.read_csv(source_path)
output_dir = Path(args.output_data)
output_dir.mkdir(parents=True, exist_ok=True)
summary = {
    "source_file": source_path.name,
    "row_count": len(frame),
    "column_count": len(frame.columns),
    "columns": frame.columns.tolist(),
}
(output_dir / "summary.json").write_text(
    json.dumps(summary, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, indent=2))
'''
script_source = textwrap.dedent(script_source).lstrip()
ast.parse(script_source, filename=str(script_path))
script_path.write_text(script_source, encoding="utf-8")
print(f"Generated command script: {script_path}")

job = command(
    display_name="Workshop taxi CSV summary",
    experiment_name=EXPERIMENT_NAME,
    code=str(generated_code_dir),
    command=(
        "python summarize.py "
        "--input-data ${{inputs.input_data}} "
        "--output-data ${{outputs.output_data}}"
    ),
    inputs={
        "input_data": Input(
            type=AssetTypes.URI_FILE,
            path=str(WORKSHOP_ROOT / "data/taxi/raw/yellowTaxiData.csv"),
        )
    },
    outputs={
        "output_data": Output(type=AssetTypes.URI_FOLDER, mode="rw_mount")
    },
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute=COMPUTE_NAME,
    identity=ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID),
    tags={"workshop": "azureml-h2o", "operation": "command-job"},
)

assert isinstance(job.identity, ManagedIdentityConfiguration)
assert job.identity.client_id == COMPUTE_IDENTITY_CLIENT_ID
print(f"Runtime identity: compute cluster UMI {COMPUTE_IDENTITY_CLIENT_ID}")

if SUBMIT:
    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted job: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Job ended with status {final_job.status}")

    output_datastore = os.getenv("AZUREML_OUTPUT_DATASTORE", "workspaceblobstore")
    output_uri = (
        f"azureml://datastores/{output_datastore}/paths/azureml/"
        f"{final_job.name}/output_data/"
    )
    download_dir = WORKSHOP_ROOT / "outputs/foundation_command_job" / final_job.name
    ml_client.jobs.download(
        final_job.name,
        output_name="output_data",
        download_path=download_dir,
    )
    summary_path = download_dir / "named-outputs/output_data/summary.json"
    if not summary_path.is_file():
        raise FileNotFoundError(f"Expected downloaded output at {summary_path}")

    print(f"Studio: {final_job.studio_url}")
    print(f"Output URI: {output_uri}")
    print(f"Downloaded summary: {summary_path}")
else:
    print(f"Prepared command job for compute: {COMPUTE_NAME}")
    print("Submission disabled. Set SUBMIT_FOUNDATION_JOB=true in workshop/.env.")

Generated command script: /mnt/batch/tasks/shared/LS_root/mounts/clusters/aml-instance-dev-cc01/code/MLOPs-AzureML-backup-a7a4bc8-20260915/workshop/outputs/generated/foundations/command_job/code/summarize.py
Runtime identity: compute cluster UMI 30fd8338-79f3-4fff-b959-c56d11672e31


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Submitted job: zen_napa_ndndskytc7
RunId: zen_napa_ndndskytc7
Web View: https://ml.azure.com/runs/zen_napa_ndndskytc7?wsid=/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25/resourcegroups/rg-aml-ws-dev-cc-01/workspaces/mlwdevcc01

Streaming user_logs/std_log.txt

/bin/bash: /azureml-envs/sklearn-1.0/lib/libtinfo.so.6: no version information available (required by /bin/bash)
{
  "source_file": "yellowTaxiData.csv",
  "row_count": 5000,
  "column_count": 22,
  "columns": [
    "vendorID",
    "tpepPickupDateTime",
    "tpepDropoffDateTime",
    "passengerCount",
    "tripDistance",
    "puLocationId",
    "doLocationId",
    "startLon",
    "startLat",
    "endLon",
    "endLat",
    "rateCodeId",
    "storeAndFwdFlag",
    "paymentType",
    "fareAmount",
    "extra",
    "mtaTax",
    "improvementSurcharge",
    "tipAmount",
    "tollsAmount",
    "totalAmount",
    "transactionID"
  ]
}

Execution Summary
RunId: zen_napa_ndndskytc7
Web View: https://ml.azure.com/runs/zen_napa_ndndsk

Studio: https://ml.azure.com/runs/zen_napa_ndndskytc7?wsid=/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25/resourcegroups/rg-aml-ws-dev-cc-01/workspaces/mlwdevcc01&tid=450bb9bd-faf4-4a7f-9fc1-f621f35fcea8
Output URI: azureml://datastores/workspaceblobstore/paths/azureml/zen_napa_ndndskytc7/output_data/
Downloaded summary: /mnt/batch/tasks/shared/LS_root/mounts/clusters/aml-instance-dev-cc01/code/MLOPs-AzureML-backup-a7a4bc8-20260915/workshop/outputs/foundation_command_job/zen_napa_ndndskytc7/named-outputs/output_data/summary.json


## Expected Result

The command job completes on the configured cluster and publishes a `summary.json` artifact in its output folder.

Next: `06_deploy_online_endpoint.ipynb`.